In [ ]:
from collections import defaultdict

In [ ]:
from utils import list_smoldoc_configs, save_dataset_dict

dataset="google/smol"
smoldoc_configs = list_smoldoc_configs(dataset)
print(f"{len(smoldoc_configs)} SmolDoc configs")

In [ ]:
from utils import get_or_build_smoldoc

datasets_dict = get_or_build_smoldoc(
    dataset_name=dataset,
    configs=smoldoc_configs,
    save_path="data/smoldoc_datasets",
    overwrite=False  # Set to True if you want to force re-download
)

In [ ]:
print(datasets_dict)

In [ ]:
# There are 102 configs, so check that they are in the same format

# List all columns across configs
feature_map = defaultdict(set)
for cfg, ds in datasets_dict.items():
    feature_map[cfg].update(ds.column_names)
# Global union of features
all_features = set().union(*feature_map.values())
print("\n=== All possible features across all configs ===")
print(all_features)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Compute row counts
row_counts = {cfg: len(ds) for cfg, ds in datasets_dict.items()}

# Build DataFrame
df_counts = (
    pd.DataFrame(list(row_counts.items()), columns=["config", "num_topics"])
    .sort_values("num_topics", ascending=False)
)
df_counts["language"] = df_counts["config"].str.extract(r"smoldoc__([a-z]{2})")

# --- Plot setup: configs on X-axis, topics on Y-axis ---
plt.figure(figsize=(18, 8))  # wide to fit labels

bars = plt.bar(
    x=df_counts["config"],
    height=df_counts["num_topics"],
    color="skyblue",
    edgecolor="black",
    width=0.8
)

plt.ylabel("Number of rows", fontsize=12)
plt.xlabel("SmolDoc Config", fontsize=12)
plt.title("Number of rows per SmolDoc Config", fontsize=14, fontweight="bold")

# Rotate and space out config labels
plt.xticks(rotation=60, ha='right', fontsize=8)
plt.subplots_adjust(bottom=0.35)  # space for long config names

# Add value labels rotated vertically
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2 + 0.2,
        height + 2,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=8,
        rotation=45
    )

plt.tight_layout()
plt.show()

In [ ]:
from utils import get_smoldoc_factuality, load_factuality_ratings

JSON_PATH = "smoldoc-factuality-ratings.json"  # adjust if needed

data = get_smoldoc_factuality()
factuality_ds = load_factuality_ratings(json_data=data)
factuality_ds

In [ ]:
keep_cols = [
    'annotator_1_label', 'annotator_1_notes',
    'annotator_2_label', 'annotator_2_notes',
    'annotator_3_label', 'annotator_3_notes'
]

# Build lookup dict: id -> annotations
annot_fields_by_id = {
    k: {col: v for col, v in zip(keep_cols, vals)}
    for k, *vals in zip(
        factuality_ds["id"],
        *[factuality_ds[col] for col in keep_cols]
    )
}

def _join_annotations(batch):
    ids = batch["id"]
    # Vectorized per-column build
    return {
        col: [annot_fields_by_id[i][col] if i in annot_fields_by_id else None for i in ids]
        for col in keep_cols
    }

# Join all configs
fact_annot_ds = datasets_dict.map(_join_annotations, batched=True)


In [ ]:
save_dataset_dict(fact_annot_ds, "data/smoldoc_factuality_joined", overwrite=True)

In [ ]:
example_cfg = 'smoldoc__en_aa'
fact_annot_ds[example_cfg]

In [ ]:
fact_annot_ds[example_cfg].features